# House Pricing Analysis with SVM Regression

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Load Datasets

In [ ]:
with open("../datasets/house_prices.pkl", mode="rb") as f:
    X_house_prices_train, X_house_prices_test, y_house_prices_train, y_house_prices_test = pickle.load(f)

print(X_house_prices_train.shape)
print(X_house_prices_test.shape)
print(y_house_prices_train.shape)
print(y_house_prices_test.shape)

## Tuning

### Grid Search

In [ ]:
pipeline_base = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

wrapper_model = TransformedTargetRegressor(
    regressor=pipeline_base,
    func=np.log1p,
    inverse_func=np.expm1
)

params = {
    'regressor__svr__kernel': ['rbf'],
    'regressor__svr__C': [0.1, 1, 10, 100],
    'regressor__svr__gamma': ['scale', 'auto', 0.01, 0.1],
    'regressor__svr__epsilon': [0.01, 0.1, 0.2]
}

grid_search = GridSearchCV(
    estimator=wrapper_model,
    param_grid=params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
).fit(
    X=X_house_prices_train,
    y=y_house_prices_train
)
print(f"Best params: {grid_search.best_params_}")
# {'regressor__svr__C': 10, 'regressor__svr__epsilon': 0.01, 'regressor__svr__gamma': 0.01, 'regressor__svr__kernel': 'rbf'}

## Train/Test

In [ ]:
pipline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(
        kernel='rbf',
        C=10.0,
        gamma=0.01,
        epsilon=0.01
    ))
])

model = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1
).fit(
    X=X_house_prices_train,
    y=y_house_prices_train
)

## Evaluation

### Metrics

In [ ]:
print(f"R²: {model.score(X=X_house_prices_train, y=y_house_prices_train)} (train set)")
print(f"R²: {model.score(X=X_house_prices_test, y=y_house_prices_test)} (test set)")

In [ ]:
y_house_prices_pred = model.predict(X_house_prices_test)

print(f"MAE: {mean_absolute_error(y_pred=y_house_prices_pred, y_true=y_house_prices_test)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_pred=y_house_prices_pred, y_true=y_house_prices_test))}")

### Visualization

In [ ]:
plt.figure(figsize=(10, 10))
plt.title(f"Estimator's Overlay of True and Predict Values for SVM Regressor")
plt.plot(y_house_prices_test[::50], label="Y Test Set")
plt.plot(y_house_prices_pred[::50], label="Y Pred Set")
plt.legend();